# Composite Harmonization GAN

**Strategy:** Instead of asking a GAN to simultaneously learn placement, appearance, and lighting,
we decompose the problem:

1. **Compositing (no learning):** Paste the furniture crop onto the empty room at the known bbox location.
   This produces a rough composite with hard edges, wrong lighting, and no shadows.

2. **Harmonization GAN (this is what we train):** A U-Net generator learns to transform the crude
   composite into a photorealistic furnished room — fixing lighting, adding shadows, blending edges.

**Architecture:**
- **Generator:** U-Net encoder-decoder (composite + mask = 4ch input, 3ch RGB output)
- **Discriminator:** PatchGAN with spectral normalization (hinge loss)
- **Loss:** Hinge GAN + 100 * L1 + 10 * VGG perceptual

**Data:** Uses existing 3D-FRONT processed triplets (input/furniture/target + metadata.csv).
No new preprocessing needed — composites are created on-the-fly during training.

**Inference pipeline:**
Room image → Mask2Former (room segmentation) → Placement heuristics → Paste furniture → Harmonization GAN → Output

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras import layers

print(f'TensorFlow version: {tf.__version__}')
print(f'GPUs available: {len(tf.config.list_physical_devices("GPU"))}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

POSSIBLE_ROOTS = [
    '/content/drive/MyDrive/processed-256',
    '/content/drive/MyDrive/693-project/data/processed-256',
    '/content/drive/MyDrive/693-project/processed-256',
    '/content/drive/MyDrive/693 project/processed-256',
]

DATA_ROOT = None
for path in POSSIBLE_ROOTS:
    if os.path.isfile(os.path.join(path, 'metadata.csv')):
        DATA_ROOT = path
        break

if DATA_ROOT is None:
    print('Dataset not found. Add the shared folder as a Drive shortcut first.')
    raise FileNotFoundError('See instructions in Notebook 01')

CSV_PATH = os.path.join(DATA_ROOT, 'metadata.csv')
print(f'Dataset root: {DATA_ROOT}')

CHECKPOINT_DIR = '/content/drive/MyDrive/693-project/checkpoints/harmonization_gan'
SAMPLE_DIR     = '/content/drive/MyDrive/693-project/samples/harmonization_gan'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

In [ ]:
import shutil, time

LOCAL_DATA = '/content/local_data'
if not os.path.isdir(LOCAL_DATA):
    print('Copying dataset to local SSD...')
    t0 = time.time()
    shutil.copytree(DATA_ROOT, LOCAL_DATA)
    print(f'Done in {time.time() - t0:.0f}s')
else:
    print(f'Local copy exists at {LOCAL_DATA}')

DATA_ROOT = LOCAL_DATA
CSV_PATH = os.path.join(DATA_ROOT, 'metadata.csv')

for split in ['train', 'val', 'test']:
    n = len(os.listdir(os.path.join(DATA_ROOT, split, 'input')))
    print(f'  {split}/input: {n} files (local SSD)')

In [ ]:
IMG_SIZE     = 256
BATCH_SIZE   = 8
LR_G         = 1e-4
LR_D         = 4e-4
BETA1        = 0.0
BETA2        = 0.999
LAMBDA_L1    = 100.0
LAMBDA_PERC  = 1.0
NUM_EPOCHS   = 200
SAVE_EPOCH   = 10
SAMPLE_EPOCH = 5
RESUME       = True

## Dataset: On-the-Fly Composite Creation with Augmentation

Each training sample is built from the existing triplets:
1. Load the empty room (`input/`)
2. Load the furniture crop (`furniture/`)
3. **Augment the composite** to force the GAN to learn real harmonization:
   - **Color jitter**: randomly shift brightness, contrast, saturation, hue of the furniture so lighting doesn't match the room
   - **Bbox perturbation**: randomly shift and scale the placement bbox so the furniture isn't perfectly aligned
   - **Cross-room furniture swap** (50% chance): use a furniture crop from a DIFFERENT room — completely different lighting, color, perspective
   - **Random horizontal flip** on the furniture crop
4. Create a binary mask from the (perturbed) bbox
5. Load the ground truth (`target/`)

These augmentations ensure the GAN sees realistic mismatches during training — not just perfect paste-backs.
Validation and test sets use NO augmentation (exact bbox, no jitter) to measure true performance.

In [ ]:
df = pd.read_csv(CSV_PATH)

def rebase_path(p):
    parts = p.replace('\\', '/').split('/')
    for i, part in enumerate(parts):
        if part in ('train', 'val', 'test'):
            return os.path.join(DATA_ROOT, *parts[i:])
    return p

for col in ['input_path', 'target_path', 'furniture_path']:
    df[col] = df[col].apply(rebase_path)

# Rescale bboxes if they were recorded at original resolution
sample_img = Image.open(df['target_path'].iloc[0])
stored_w, stored_h = sample_img.size
bbox_max = max(df['bbox_x2'].max(), df['bbox_y2'].max())

if bbox_max > max(stored_w, stored_h):
    orig_w = int(np.ceil(df['bbox_x2'].max() / 256) * 256)
    orig_h = int(np.ceil(df['bbox_y2'].max() / 256) * 256)
    for c in ['bbox_x1', 'bbox_x2']:
        df[c] *= stored_w / orig_w
    for c in ['bbox_y1', 'bbox_y2']:
        df[c] *= stored_h / orig_h
    print(f'Rescaled bboxes from {orig_w}x{orig_h} to {stored_w}x{stored_h}')

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

In [ ]:
from PIL import ImageEnhance
import random as _random

# Build a lookup table for cross-room furniture swapping during training.
# Each training sample has a 50% chance of using furniture from a DIFFERENT room,
# which forces the GAN to handle real lighting/color mismatches.
_swap_furn_paths = train_df['furniture_path'].values.copy()


def _jitter_color(pil_img):
    """Mildly adjust brightness, contrast, and saturation."""
    pil_img = ImageEnhance.Brightness(pil_img).enhance(_random.uniform(0.9, 1.1))
    pil_img = ImageEnhance.Contrast(pil_img).enhance(_random.uniform(0.9, 1.1))
    pil_img = ImageEnhance.Color(pil_img).enhance(_random.uniform(0.85, 1.15))
    return pil_img


def _perturb_bbox(x1, y1, x2, y2, img_size):
    """Randomly shift and scale the bbox to break perfect alignment."""
    w, h = x2 - x1, y2 - y1
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2

    # Random shift: up to 15% of bbox size in any direction
    cx += _random.uniform(-0.15, 0.15) * w
    cy += _random.uniform(-0.15, 0.15) * h

    # Random scale: 80% to 120% of original size
    scale_f = _random.uniform(0.8, 1.2)
    w *= scale_f
    h *= scale_f

    nx1 = max(0, int(cx - w / 2))
    ny1 = max(0, int(cy - h / 2))
    nx2 = min(img_size, int(cx + w / 2))
    ny2 = min(img_size, int(cy + h / 2))

    if nx2 - nx1 < 8:
        nx2 = min(img_size, nx1 + 8)
    if ny2 - ny1 < 8:
        ny2 = min(img_size, ny1 + 8)

    return nx1, ny1, nx2, ny2


def create_composite_and_mask_np(room_path, furn_path, bx1, by1, bx2, by2,
                                  img_size=256, augment=True):
    """Create a rough composite and binary mask with optional augmentation.

    Augmentations (training only):
    - Color jitter on furniture (brightness, contrast, saturation)
    - Bbox position/scale perturbation
    - Random horizontal flip of the furniture crop
    - 50% chance: swap furniture with a crop from a different room

    Returns (composite, mask, room, furniture) as float32 arrays in [-1, 1],
    except mask which is in [0, 1].
    """
    room_pil = Image.open(room_path).convert('RGB').resize((img_size, img_size), Image.BILINEAR)
    furn_pil = Image.open(furn_path).convert('RGB')

    # Scale bbox to target resolution
    scale = img_size / max(Image.open(room_path).size)
    x1 = max(0, int(bx1 * scale))
    y1 = max(0, int(by1 * scale))
    x2 = min(img_size, int(bx2 * scale))
    y2 = min(img_size, int(by2 * scale))

    w, h = x2 - x1, y2 - y1
    if w < 4 or h < 4:
        w, h = max(w, 4), max(h, 4)
        x2, y2 = min(x1 + w, img_size), min(y1 + h, img_size)

    if augment:
        # Mild color jitter on the furniture
        furn_pil = _jitter_color(furn_pil)

        # Random horizontal flip
        if _random.random() < 0.5:
            furn_pil = furn_pil.transpose(Image.FLIP_LEFT_RIGHT)

    furn_resized = furn_pil.resize((x2 - x1, y2 - y1), Image.BILINEAR)

    composite = room_pil.copy()
    composite.paste(furn_resized, (x1, y1))

    mask = np.zeros((img_size, img_size, 1), dtype=np.float32)
    mask[y1:y2, x1:x2, 0] = 1.0

    composite_np = np.array(composite, dtype=np.float32) / 127.5 - 1.0
    room_np      = np.array(room_pil, dtype=np.float32) / 127.5 - 1.0
    furn_np      = np.array(furn_pil.resize((img_size, img_size), Image.BILINEAR),
                            dtype=np.float32) / 127.5 - 1.0

    return composite_np, mask, room_np, furn_np


def load_target_np(target_path, img_size=256):
    target_pil = Image.open(target_path).convert('RGB').resize((img_size, img_size), Image.BILINEAR)
    return np.array(target_pil, dtype=np.float32) / 127.5 - 1.0

In [ ]:
def _make_py_load_fn(augment):
    """Create a py_function-compatible loader with the augment flag baked in."""
    def py_load_sample(input_path, target_path, furniture_path, bx1, by1, bx2, by2):
        room_p = input_path.numpy().decode('utf-8')
        targ_p = target_path.numpy().decode('utf-8')
        furn_p = furniture_path.numpy().decode('utf-8')

        composite, mask, room, furn = create_composite_and_mask_np(
            room_p, furn_p,
            bx1.numpy(), by1.numpy(), bx2.numpy(), by2.numpy(),
            img_size=IMG_SIZE, augment=augment
        )
        target = load_target_np(targ_p, img_size=IMG_SIZE)

        return composite, mask, target, room, furn
    return py_load_sample


def _make_tf_load_fn(augment):
    """tf.data-compatible wrapper with augment flag."""
    py_fn = _make_py_load_fn(augment)

    def tf_load_sample(input_path, target_path, furniture_path, bx1, by1, bx2, by2):
        composite, mask, target, room, furn = tf.py_function(
            py_fn,
            [input_path, target_path, furniture_path, bx1, by1, bx2, by2],
            [tf.float32, tf.float32, tf.float32, tf.float32, tf.float32]
        )
        composite.set_shape([IMG_SIZE, IMG_SIZE, 3])
        mask.set_shape([IMG_SIZE, IMG_SIZE, 1])
        target.set_shape([IMG_SIZE, IMG_SIZE, 3])
        room.set_shape([IMG_SIZE, IMG_SIZE, 3])
        furn.set_shape([IMG_SIZE, IMG_SIZE, 3])
        return composite, mask, target, room, furn
    return tf_load_sample


def make_dataset(split_df, shuffle=True, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((
        split_df['input_path'].values,
        split_df['target_path'].values,
        split_df['furniture_path'].values,
        split_df['bbox_x1'].values.astype(np.float32),
        split_df['bbox_y1'].values.astype(np.float32),
        split_df['bbox_x2'].values.astype(np.float32),
        split_df['bbox_y2'].values.astype(np.float32),
    ))
    if shuffle:
        ds = ds.shuffle(len(split_df))
    ds = ds.map(_make_tf_load_fn(augment), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


# All augmentation OFF for now — let the model learn clean pass-through first.
# Once sharp outputs are confirmed, augmentation can be re-enabled for fine-tuning.
train_ds = make_dataset(train_df, shuffle=True, augment=False)
val_ds   = make_dataset(val_df, shuffle=False, augment=False)
test_ds  = make_dataset(test_df, shuffle=False, augment=False)

In [ ]:
# Visualize AUGMENTED training composites vs clean validation composites
# This shows the effect of color jitter, bbox perturbation, and cross-room swapping.

fig, axes = plt.subplots(5, 8, figsize=(32, 20))
row_labels = ['Empty Room', 'Furniture (aug)', 'Composite (aug)', 'Mask', 'Ground Truth']

# Left 4 columns: augmented TRAINING samples
for composite, mask, target, room, furn in train_ds.take(1):
    for i in range(min(4, composite.shape[0])):
        axes[0, i].imshow((room[i].numpy() + 1) / 2)
        axes[1, i].imshow((furn[i].numpy() + 1) / 2)
        axes[2, i].imshow((composite[i].numpy() + 1) / 2)
        axes[3, i].imshow(mask[i].numpy().squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[4, i].imshow((target[i].numpy() + 1) / 2)
        if i == 0:
            for j in range(5):
                axes[j, i].set_ylabel(row_labels[j], fontsize=11, rotation=90, labelpad=12)

# Right 4 columns: clean VALIDATION samples (no augmentation)
for composite, mask, target, room, furn in val_ds.take(1):
    for i in range(min(4, composite.shape[0])):
        col = i + 4
        axes[0, col].imshow((room[i].numpy() + 1) / 2)
        axes[1, col].imshow((furn[i].numpy() + 1) / 2)
        axes[2, col].imshow((composite[i].numpy() + 1) / 2)
        axes[3, col].imshow(mask[i].numpy().squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[4, col].imshow((target[i].numpy() + 1) / 2)

for j in range(5):
    for i in range(8):
        axes[j, i].axis('off')

# Column group titles
axes[0, 1].set_title('TRAINING (augmented)', fontsize=14, fontweight='bold', color='red')
axes[0, 5].set_title('VALIDATION (clean)', fontsize=14, fontweight='bold', color='green')

plt.suptitle('Augmented vs Clean Composites: Training forces the GAN to handle mismatches',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## Model Architecture

**Generator:** Standard U-Net (same as Pix2Pix) with 4-channel input (composite RGB + mask).
The mask channel explicitly tells the generator where harmonization is needed.

**Discriminator:** PatchGAN with spectral normalization for stable training.
Receives the composite as condition + real/fake as target.

In [ ]:
WEIGHT_INIT = tf.keras.initializers.RandomNormal(stddev=0.02)


class InstanceNormalization(layers.Layer):
    def __init__(self, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        ch = input_shape[-1]
        self.scale  = self.add_weight(name='scale',  shape=(ch,), initializer='ones')
        self.offset = self.add_weight(name='offset', shape=(ch,), initializer='zeros')

    def call(self, x):
        mean, var = tf.nn.moments(x, axes=[1, 2], keepdims=True)
        return self.scale * (x - mean) / tf.sqrt(var + self.epsilon) + self.offset


def downsample(filters, size=4, apply_norm=True, dropout=0.0):
    block = tf.keras.Sequential()
    block.add(layers.Conv2D(filters, size, strides=2, padding='same',
                            kernel_initializer=WEIGHT_INIT, use_bias=False))
    if apply_norm:
        block.add(InstanceNormalization())
    block.add(layers.LeakyReLU(0.2))
    if dropout > 0:
        block.add(layers.Dropout(dropout))
    return block


def upsample(filters, size=4, dropout=0.0):
    block = tf.keras.Sequential()
    block.add(layers.Conv2DTranspose(filters, size, strides=2, padding='same',
                                     kernel_initializer=WEIGHT_INIT, use_bias=False))
    block.add(InstanceNormalization())
    block.add(layers.ReLU())
    if dropout > 0:
        block.add(layers.Dropout(dropout))
    return block

In [ ]:
def build_generator(in_channels=4):
    """U-Net generator: composite(3ch) + mask(1ch) -> harmonized image(3ch)."""
    inputs = layers.Input(shape=[IMG_SIZE, IMG_SIZE, in_channels])

    down_stack = [
        downsample(64, apply_norm=False),        # 128x128
        downsample(128),                          # 64x64
        downsample(256),                          # 32x32
        downsample(512, dropout=0.5),             # 16x16
        downsample(512, dropout=0.5),             # 8x8
        downsample(512, dropout=0.5),             # 4x4
        downsample(512, dropout=0.5),             # 2x2
        downsample(512, apply_norm=False, dropout=0.5),  # 1x1
    ]

    up_stack = [
        upsample(512, dropout=0.5),   # 2x2
        upsample(512, dropout=0.5),   # 4x4
        upsample(512, dropout=0.5),   # 8x8
        upsample(512, dropout=0.5),   # 16x16
        upsample(256),                # 32x32
        upsample(128),                # 64x64
        upsample(64),                 # 128x128
    ]

    last = layers.Conv2DTranspose(
        3, 4, strides=2, padding='same',
        kernel_initializer=WEIGHT_INIT, activation='tanh'
    )

    x = inputs
    skips = []
    for down in down_stack:
        x = down(x)
        skips.append(x)

    skips = list(reversed(skips[:-1]))

    for up, skip in zip(up_stack, skips):
        x = up(x)
        x = layers.Concatenate()([x, skip])

    outputs = last(x)
    return tf.keras.Model(inputs, outputs, name='generator')

In [ ]:
SN = layers.SpectralNormalization


def build_discriminator():
    """PatchGAN discriminator with spectral normalization.

    Input: condition (composite + mask = 4ch) + image (real or fake = 3ch) = 7ch total.
    Output: 30x30 patch map.
    """
    cond = layers.Input(shape=[IMG_SIZE, IMG_SIZE, 4], name='condition')
    tar  = layers.Input(shape=[IMG_SIZE, IMG_SIZE, 3], name='target')
    x = layers.Concatenate()([cond, tar])  # 7 channels

    x = SN(layers.Conv2D(64, 4, strides=2, padding='same',
                          kernel_initializer=WEIGHT_INIT))(x)
    x = layers.LeakyReLU(0.2)(x)           # 128x128

    x = SN(layers.Conv2D(128, 4, strides=2, padding='same',
                          kernel_initializer=WEIGHT_INIT))(x)
    x = layers.LeakyReLU(0.2)(x)           # 64x64

    x = SN(layers.Conv2D(256, 4, strides=2, padding='same',
                          kernel_initializer=WEIGHT_INIT))(x)
    x = layers.LeakyReLU(0.2)(x)           # 32x32

    x = layers.ZeroPadding2D()(x)          # 34x34
    x = SN(layers.Conv2D(512, 4, strides=1,
                          kernel_initializer=WEIGHT_INIT))(x)
    x = layers.LeakyReLU(0.2)(x)           # 31x31

    x = layers.ZeroPadding2D()(x)          # 33x33
    x = SN(layers.Conv2D(1, 4, strides=1,
                          kernel_initializer=WEIGHT_INIT))(x)  # 30x30

    return tf.keras.Model(inputs=[cond, tar], outputs=x, name='discriminator')

In [ ]:
def build_vgg_features():
    """VGG19 feature extractor for perceptual loss."""
    vgg = tf.keras.applications.VGG19(include_top=False, weights='imagenet')
    vgg.trainable = False
    layer_names = ['block1_conv1', 'block2_conv1', 'block3_conv1', 'block4_conv1']
    outputs = [vgg.get_layer(n).output for n in layer_names]
    return tf.keras.Model(vgg.input, outputs, name='vgg_features')


def perceptual_loss(vgg, fake, real):
    """L1 distance between VGG19 intermediate features."""
    fake_pp = tf.keras.applications.vgg19.preprocess_input((fake + 1.0) * 127.5)
    real_pp = tf.keras.applications.vgg19.preprocess_input((real + 1.0) * 127.5)
    fake_feats = vgg(fake_pp)
    real_feats = vgg(real_pp)
    return tf.add_n([tf.reduce_mean(tf.abs(f - r)) for f, r in zip(fake_feats, real_feats)])

## Initialize Models

In [ ]:
import json as _json

generator     = build_generator(in_channels=4)
discriminator = build_discriminator()
vgg           = build_vgg_features()

opt_g = tf.keras.optimizers.Adam(learning_rate=LR_G, beta_1=BETA1, beta_2=BETA2)
opt_d = tf.keras.optimizers.Adam(learning_rate=LR_D, beta_1=BETA1, beta_2=BETA2)

checkpoint = tf.train.Checkpoint(
    generator=generator,
    discriminator=discriminator,
    opt_g=opt_g,
    opt_d=opt_d,
)
ckpt_manager = tf.train.CheckpointManager(checkpoint, CHECKPOINT_DIR, max_to_keep=5)

HISTORY_PATH = os.path.join(CHECKPOINT_DIR, 'history.json')
START_EPOCH = 1
saved_history = None

if RESUME and ckpt_manager.latest_checkpoint:
    checkpoint.restore(ckpt_manager.latest_checkpoint)
    ckpt_name = os.path.basename(ckpt_manager.latest_checkpoint)
    ckpt_num = int(ckpt_name.split('-')[-1])
    START_EPOCH = ckpt_num * SAVE_EPOCH + 1
    print(f'Restored from {ckpt_manager.latest_checkpoint} -> resuming at epoch {START_EPOCH}')
    if os.path.isfile(HISTORY_PATH):
        try:
            with open(HISTORY_PATH) as f:
                saved_history = _json.load(f)
            print(f'Loaded loss history ({len(saved_history["g_loss"])} epochs)')
        except _json.JSONDecodeError:
            print('Warning: Could not load history. Starting new history.')
            saved_history = None
else:
    print('Starting fresh (no checkpoint found or RESUME=False)')

print(f'\nGenerator params:     {generator.count_params():,}')
print(f'Discriminator params: {discriminator.count_params():,}')
generator.summary()

## Training

**Losses:**
- Discriminator: Hinge loss with spectral norm (stable, no mode collapse)
- Generator: Hinge adversarial + L1 reconstruction + VGG perceptual

**TTUR (Two Time-scale Update Rule):** D learns 4x faster than G to stay ahead.

In [ ]:
@tf.function
def train_step(composite, mask, target):
    gen_input = tf.concat([composite, mask], axis=-1)  # 4 channels

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        fake = generator(gen_input, training=True)

        disc_real = discriminator([gen_input, target], training=True)
        disc_fake = discriminator([gen_input, fake], training=True)

        # Hinge loss (discriminator)
        d_loss = 0.5 * (tf.reduce_mean(tf.nn.relu(1.0 - disc_real)) +
                        tf.reduce_mean(tf.nn.relu(1.0 + disc_fake)))

        # Hinge loss (generator)
        g_loss_adv = -tf.reduce_mean(disc_fake)

        # L1 reconstruction
        g_loss_l1 = tf.reduce_mean(tf.abs(fake - target))

        # VGG perceptual
        g_loss_perc = perceptual_loss(vgg, fake, target)

        g_loss = g_loss_adv + LAMBDA_L1 * g_loss_l1 + LAMBDA_PERC * g_loss_perc

    gen_grads  = gen_tape.gradient(g_loss, generator.trainable_variables)
    disc_grads = disc_tape.gradient(d_loss, discriminator.trainable_variables)

    opt_g.apply_gradients(zip(gen_grads, generator.trainable_variables))
    opt_d.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    return g_loss, d_loss, g_loss_l1, g_loss_perc

In [ ]:
def save_samples(epoch, n=4):
    for composite, mask, target, room, furn in val_ds.take(1):
        gen_input = tf.concat([composite[:n], mask[:n]], axis=-1)
        fake = generator(gen_input, training=False)

        comp_vis   = np.clip((composite[:n].numpy() + 1) / 2, 0, 1)
        mask_vis   = mask[:n].numpy().squeeze(-1)
        fake_vis   = np.clip((fake.numpy() + 1) / 2, 0, 1)
        target_vis = (target[:n].numpy() + 1) / 2

    fig, axes = plt.subplots(4, n, figsize=(4 * n, 16))
    for i in range(n):
        axes[0, i].imshow(comp_vis[i])
        axes[0, i].set_title('Composite')
        axes[0, i].axis('off')
        axes[1, i].imshow(mask_vis[i], cmap='gray')
        axes[1, i].set_title('Mask')
        axes[1, i].axis('off')
        axes[2, i].imshow(fake_vis[i])
        axes[2, i].set_title('Harmonized')
        axes[2, i].axis('off')
        axes[3, i].imshow(target_vis[i])
        axes[3, i].set_title('Ground Truth')
        axes[3, i].axis('off')
    plt.suptitle(f'Epoch {epoch}', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(SAMPLE_DIR, f'epoch_{epoch:04d}.png'), dpi=100)
    plt.show()
    plt.close()


def update_lr(optimizer, epoch, initial_lr, num_epochs):
    decay_start = num_epochs // 2
    if epoch >= decay_start:
        new_lr = initial_lr * (1.0 - (epoch - decay_start) / (num_epochs - decay_start))
        optimizer.learning_rate.assign(max(new_lr, 1e-7))

In [ ]:
if saved_history is not None:
    history = saved_history
else:
    history = {'g_loss': [], 'd_loss': [], 'l1': [], 'perc': [], 'val_l1': []}

steps_per_epoch = len(train_df) // BATCH_SIZE
print(f'Training from epoch {START_EPOCH} to {NUM_EPOCHS}')

for epoch in range(START_EPOCH, NUM_EPOCHS + 1):
    eg, ed, el, ep = 0.0, 0.0, 0.0, 0.0
    nb = 0

    pbar = tqdm(train_ds, total=steps_per_epoch, desc=f'Epoch {epoch}/{NUM_EPOCHS}')
    for composite, mask, target, _, _ in pbar:
        g, d, l1, perc = train_step(composite, mask, target)
        eg += g.numpy(); ed += d.numpy(); el += l1.numpy(); ep += perc.numpy()
        nb += 1
        pbar.set_postfix({'G': f'{g.numpy():.3f}', 'D': f'{d.numpy():.3f}'})

    history['g_loss'].append(float(eg / nb))
    history['d_loss'].append(float(ed / nb))
    history['l1'].append(float(el / nb))
    history['perc'].append(float(ep / nb))

    # Validation L1
    val_l1 = 0.0
    nv = 0
    for vc, vm, vt, _, _ in val_ds:
        vgen_input = tf.concat([vc, vm], axis=-1)
        vfake = generator(vgen_input, training=False)
        val_l1 += tf.reduce_mean(tf.abs(vfake - vt)).numpy()
        nv += 1
    history['val_l1'].append(float(val_l1 / max(nv, 1)))

    update_lr(opt_g, epoch, LR_G, NUM_EPOCHS)
    update_lr(opt_d, epoch, LR_D, NUM_EPOCHS)

    print(f'Epoch {epoch} | G: {history["g_loss"][-1]:.3f} | D: {history["d_loss"][-1]:.3f} | '
          f'L1: {history["l1"][-1]:.4f} | Perc: {history["perc"][-1]:.2f} | '
          f'Val L1: {history["val_l1"][-1]:.4f}')

    if epoch % SAMPLE_EPOCH == 0:
        save_samples(epoch)
    if epoch % SAVE_EPOCH == 0:
        ckpt_manager.save()
        with open(HISTORY_PATH, 'w') as f:
            _json.dump(history, f)
        print(f'  Checkpoint + history saved')

ckpt_manager.save()
with open(HISTORY_PATH, 'w') as f:
    _json.dump(history, f)
print('Training complete.')

## Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 4))
for ax, key, title in zip(axes, history.keys(),
        ['Generator', 'Discriminator', 'Train L1', 'Perceptual', 'Val L1']):
    ax.plot(history[key])
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'loss_curves.png'), dpi=150)
plt.show()

## Evaluation

Metrics on the held-out test set:

| Metric | What it measures | Direction |
|--------|-----------------|-----------|
| **SSIM** | Structural similarity to ground truth (full image) | Higher = better |
| **PSNR** | Pixel-level reconstruction quality (full image) | Higher = better |
| **FID** | Distribution-level perceptual quality | Lower = better |
| **LPIPS** | Learned perceptual distance (AlexNet features) | Lower = better |
| **BG-PSNR** | Background preservation (pixels outside mask) | Higher = better |
| **fMSE** | Foreground MSE (harmonization quality in furniture region only) | Lower = better |

BG-PSNR and fMSE together tell the full story: did the GAN harmonize the furniture region (low fMSE)
without degrading the background (high BG-PSNR)?

In [ ]:
all_ssim, all_psnr, all_bg_psnr, all_fmse, all_lpips = [], [], [], [], []

fid_real_dir = os.path.join(SAMPLE_DIR, 'fid_real')
fid_fake_dir = os.path.join(SAMPLE_DIR, 'fid_fake')
os.makedirs(fid_real_dir, exist_ok=True)
os.makedirs(fid_fake_dir, exist_ok=True)

# LPIPS: uses a pre-trained AlexNet to measure perceptual distance.
# We compute it in TF by extracting VGG features (same network we used for training).
# For a more standard LPIPS we install the lpips package separately below.

img_idx = 0
for composite, mask, target, _, _ in tqdm(test_ds, desc='Evaluating'):
    gen_input = tf.concat([composite, mask], axis=-1)
    fake = generator(gen_input, training=False)

    fake_01   = tf.clip_by_value((fake + 1.0) / 2.0, 0.0, 1.0)
    target_01 = (target + 1.0) / 2.0

    # Full-image SSIM and PSNR
    ssim_vals = tf.image.ssim(fake_01, target_01, max_val=1.0)
    psnr_vals = tf.image.psnr(fake_01, target_01, max_val=1.0)
    all_ssim.extend(ssim_vals.numpy().tolist())
    all_psnr.extend(psnr_vals.numpy().tolist())

    # VGG-based perceptual distance (proxy for LPIPS)
    perc_dist = perceptual_loss(vgg, fake, target)
    all_lpips.append(perc_dist.numpy() / fake_01.shape[0])

    bg_mask = 1.0 - mask
    fg_mask = mask

    for j in range(fake_01.shape[0]):
        bg = bg_mask[j]
        fg = fg_mask[j]

        # Background PSNR (pixels outside the mask — should be preserved)
        if tf.reduce_sum(bg) > 0:
            bg_fake   = fake_01[j] * bg
            bg_target = target_01[j] * bg
            mse = tf.reduce_sum(tf.square(bg_fake - bg_target)) / tf.reduce_sum(bg) / 3.0
            bg_psnr = -10.0 * tf.math.log(mse + 1e-10) / tf.math.log(10.0)
            all_bg_psnr.append(bg_psnr.numpy())

        # Foreground MSE (pixels inside the mask — harmonization quality)
        if tf.reduce_sum(fg) > 0:
            fg_fake   = fake_01[j] * fg
            fg_target = target_01[j] * fg
            fmse = tf.reduce_sum(tf.square(fg_fake - fg_target)) / tf.reduce_sum(fg) / 3.0
            all_fmse.append(fmse.numpy())

        # Save images for FID computation
        real_img = tf.cast(tf.clip_by_value(target_01[j] * 255.0, 0, 255), tf.uint8)
        fake_img = tf.cast(tf.clip_by_value(fake_01[j] * 255.0, 0, 255), tf.uint8)
        tf.io.write_file(
            os.path.join(fid_real_dir, f'{img_idx:05d}.png'),
            tf.image.encode_png(real_img)
        )
        tf.io.write_file(
            os.path.join(fid_fake_dir, f'{img_idx:05d}.png'),
            tf.image.encode_png(fake_img)
        )
        img_idx += 1

print('\n' + '=' * 55)
print('    Harmonization GAN — Test Set Metrics')
print('=' * 55)
print(f'  SSIM:          {np.mean(all_ssim):.4f} +/- {np.std(all_ssim):.4f}  (higher = better)')
print(f'  PSNR:          {np.mean(all_psnr):.2f} +/- {np.std(all_psnr):.2f} dB  (higher = better)')
print(f'  LPIPS (VGG):   {np.mean(all_lpips):.4f} +/- {np.std(all_lpips):.4f}  (lower = better)')
print(f'  BG-PSNR:       {np.mean(all_bg_psnr):.2f} +/- {np.std(all_bg_psnr):.2f} dB  (background preservation)')
print(f'  fMSE:          {np.mean(all_fmse):.6f} +/- {np.std(all_fmse):.6f}  (foreground harmonization)')
print('=' * 55)
print(f'  Images saved:  {img_idx} (for FID computation below)')

In [ ]:
!pip install pytorch-fid -q
!python -m pytorch_fid {fid_real_dir} {fid_fake_dir}

## Final Results Visualization

In [ ]:
n_compare = 8

for composite, mask, target, room, furn in test_ds.take(1):
    n_show = min(n_compare, composite.shape[0])
    gen_input = tf.concat([composite[:n_show], mask[:n_show]], axis=-1)
    fake = generator(gen_input, training=False)

    comp_vis   = np.clip((composite[:n_show].numpy() + 1) / 2, 0, 1)
    fake_vis   = np.clip((fake.numpy() + 1) / 2, 0, 1)
    target_vis = (target[:n_show].numpy() + 1) / 2
    room_vis   = (room[:n_show].numpy() + 1) / 2
    furn_vis   = (furn[:n_show].numpy() + 1) / 2

fig, axes = plt.subplots(n_show, 5, figsize=(25, 4.5 * n_show))
col_titles = ['Empty Room', 'Furniture Ref', 'Rough Composite', 'Harmonized (Ours)', 'Ground Truth']

for i in range(n_show):
    axes[i, 0].imshow(room_vis[i])
    axes[i, 1].imshow(furn_vis[i])
    axes[i, 2].imshow(comp_vis[i])
    axes[i, 3].imshow(fake_vis[i])
    axes[i, 4].imshow(target_vis[i])

    ssim_val = tf.image.ssim(
        tf.constant(fake_vis[i:i+1]), tf.constant(target_vis[i:i+1]), max_val=1.0
    ).numpy()[0]
    psnr_val = tf.image.psnr(
        tf.constant(fake_vis[i:i+1]), tf.constant(target_vis[i:i+1]), max_val=1.0
    ).numpy()[0]
    axes[i, 3].set_xlabel(f'SSIM: {ssim_val:.3f} | PSNR: {psnr_val:.1f} dB', fontsize=10)

    for j in range(5):
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=14, fontweight='bold')

plt.suptitle('Composite Harmonization GAN — Test Results',
             fontsize=18, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'final_results.png'), dpi=150, bbox_inches='tight')
plt.show()

---

## Inference Demo: End-to-End with Mask2Former

At inference time, for a **new room** and a **new furniture image**, we use:
1. **Mask2Former** (pre-trained on ADE20K) to segment the room into floor, walls, etc.
2. **Heuristic placement** to pick a valid location based on room structure.
3. **Compositing** to paste the furniture at the predicted location.
4. **Our trained Harmonization GAN** to make the composite photorealistic.

This section requires `transformers` and a PyTorch runtime alongside TensorFlow.

In [ ]:
!pip install -q transformers torch torchvision

In [ ]:
import torch
from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor

m2f_processor = Mask2FormerImageProcessor.from_pretrained(
    'facebook/mask2former-swin-large-ade-semantic'
)
m2f_model = Mask2FormerForUniversalSegmentation.from_pretrained(
    'facebook/mask2former-swin-large-ade-semantic'
)
m2f_model.eval()
if torch.cuda.is_available():
    m2f_model = m2f_model.cuda()
print('Mask2Former loaded (ADE20K semantic segmentation).')

In [ ]:
# ADE20K class IDs for placement logic
ADE20K_FLOOR = 3      # floor/flooring
ADE20K_WALL  = 0      # wall
ADE20K_BED   = 7      # bed


def segment_room(image_pil):
    """Run Mask2Former semantic segmentation on a room image.
    Returns a 2D numpy array of class IDs (H, W).
    """
    inputs = m2f_processor(images=image_pil, return_tensors='pt')
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = m2f_model(**inputs)

    pred = m2f_processor.post_process_semantic_segmentation(
        outputs, target_sizes=[image_pil.size[::-1]]
    )[0]
    return pred.cpu().numpy()


def find_placement_bbox(seg_map, furniture_type='bed', img_size=256):
    """Find a plausible placement location based on room segmentation.

    Strategy:
    - Identify floor pixels (where furniture can stand)
    - Identify wall pixels (furniture is typically placed against walls)
    - Find the largest floor region adjacent to a wall
    - Return a bbox sized proportionally to the room
    """
    floor_mask = (seg_map == ADE20K_FLOOR).astype(np.uint8)
    wall_mask  = (seg_map == ADE20K_WALL).astype(np.uint8)

    h, w = seg_map.shape

    if floor_mask.sum() < 100:
        # Fallback: center of the image, reasonable default size
        fw, fh = int(w * 0.4), int(h * 0.3)
        cx, cy = w // 2, int(h * 0.65)
        return (cx - fw // 2, cy - fh // 2, cx + fw // 2, cy + fh // 2)

    # Find floor rows (vertical extent of floor)
    floor_rows = np.where(floor_mask.sum(axis=1) > w * 0.1)[0]
    if len(floor_rows) == 0:
        floor_rows = np.array([int(h * 0.5), int(h * 0.8)])

    floor_top = floor_rows.min()
    floor_bot = floor_rows.max()

    # Furniture dimensions relative to floor area
    if furniture_type == 'bed':
        fw = int(w * 0.35)
        fh = int((floor_bot - floor_top) * 0.5)
    elif furniture_type in ('sofa', 'couch'):
        fw = int(w * 0.4)
        fh = int((floor_bot - floor_top) * 0.35)
    else:
        fw = int(w * 0.25)
        fh = int((floor_bot - floor_top) * 0.3)

    fw = max(fw, 30)
    fh = max(fh, 30)

    # Try to place against a wall: find wall-adjacent floor columns
    wall_cols = np.where(wall_mask.sum(axis=0) > h * 0.2)[0]
    if len(wall_cols) > 0:
        # Place near the wall center
        wall_center = int(np.median(wall_cols))
        cx = np.clip(wall_center, fw // 2, w - fw // 2)
    else:
        cx = w // 2

    cy = int(floor_top + (floor_bot - floor_top) * 0.4)
    cy = np.clip(cy, fh // 2, h - fh // 2)

    x1 = max(0, cx - fw // 2)
    y1 = max(0, cy - fh // 2)
    x2 = min(w, x1 + fw)
    y2 = min(h, y1 + fh)

    return (x1, y1, x2, y2)

In [ ]:
def inference_pipeline(room_pil, furniture_pil, furniture_type='bed'):
    """Full end-to-end inference: room + furniture -> staged room.

    1. Segment the room with Mask2Former
    2. Find a placement location based on room layout
    3. Paste the furniture at that location
    4. Run the harmonization GAN
    """
    room_resized = room_pil.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

    # Step 1: Segment the room
    seg_map = segment_room(room_resized)

    # Step 2: Find placement
    bbox = find_placement_bbox(seg_map, furniture_type, IMG_SIZE)
    x1, y1, x2, y2 = bbox

    # Step 3: Create composite
    furn_resized = furniture_pil.resize((x2 - x1, y2 - y1), Image.BILINEAR)
    composite = room_resized.copy()
    composite.paste(furn_resized, (x1, y1))

    # Create mask
    mask_np = np.zeros((IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
    mask_np[y1:y2, x1:x2, 0] = 1.0

    # Normalize
    comp_np = np.array(composite, dtype=np.float32) / 127.5 - 1.0

    # Step 4: Harmonize with our GAN
    gen_input = tf.concat([
        tf.constant(comp_np[np.newaxis]),
        tf.constant(mask_np[np.newaxis])
    ], axis=-1)
    harmonized = generator(gen_input, training=False)
    harmonized_np = np.clip((harmonized[0].numpy() + 1) / 2, 0, 1)

    return {
        'room': room_resized,
        'segmentation': seg_map,
        'bbox': bbox,
        'composite': composite,
        'mask': mask_np.squeeze(),
        'harmonized': harmonized_np,
    }

In [ ]:
# Demo: Run inference on test samples (using the empty rooms as "new" rooms)
n_demo = 4
demo_rows = test_df.sample(n_demo, random_state=42)

fig, axes = plt.subplots(n_demo, 5, figsize=(25, 5 * n_demo))
col_titles = ['Input Room', 'Room Segmentation', 'Rough Composite', 'Harmonized', 'Ground Truth']

for i, (_, row) in enumerate(demo_rows.iterrows()):
    room_pil = Image.open(row['input_path']).convert('RGB')
    furn_pil = Image.open(row['furniture_path']).convert('RGB')
    target_pil = Image.open(row['target_path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE))

    result = inference_pipeline(room_pil, furn_pil, furniture_type='bed')

    axes[i, 0].imshow(result['room'])
    axes[i, 1].imshow(result['segmentation'], cmap='tab20')
    axes[i, 2].imshow(result['composite'])
    axes[i, 3].imshow(result['harmonized'])
    axes[i, 4].imshow(target_pil)

    for j in range(5):
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=14, fontweight='bold')

plt.suptitle('End-to-End Inference: Room Segmentation -> Placement -> Harmonization',
             fontsize=16, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'inference_demo.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Inference demo complete.')